# Database

> SQLite persistence APIs for monitoring sessions, foreground activity, and process events.

This module owns the application's database connection, schema definitions,
event-writing functions, session lifecycle operations, retention cleanup, and
queries used by the reporting layer.

Timestamps are stored as ISO-formatted strings. Foreground events are the
primary source for attention reports; process events are retained as supporting
and diagnostic data.

In [ ]:
#| default_exp db

In [ ]:
#| hide
from nbdev.showdoc import *

## Schema models

In [ ]:
#| export
from datetime import datetime, timedelta
from fastlite import *

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import snooper_pkg.config as cf

In [ ]:
#| exports
class ProcessEvent:
    "A recorded process start or stop event."

    session_id: str  # Monitoring session containing the event
    pid: int         # Operating-system process identifier
    app: str         # Process or application name
    event_type: str  # Event type: `start` or `stop`
    timestamp: str   # ISO-formatted event timestamp
    id: int | None = None  # Database-generated primary key

In [ ]:
#| exports
class Session:
    "A single continuous monitoring session."

    session_id: str       # Unique session identifier
    start_time: str       # ISO-formatted session start time
    end_time: str | None  # ISO-formatted end time; `None` while open
    status: str           # Session state: `open` or `closed`
    end_reason: str | None  # Reason the session ended

In [ ]:
#| exports
class ForegroundEvent:
    "An interval during which an application or idle state was in the foreground."

    session_id: str       # Monitoring session containing the interval
    app: str              # Foreground application name
    pid: int              # Foreground process identifier
    window_title: str     # Foreground window title
    start_time: str       # ISO-formatted interval start time
    end_time: str         # ISO-formatted interval end time
    hwnd: int             # Windows window handle
    is_idle: bool         # Whether the interval represents user idle time
    id: int | None = None  # Database-generated primary key

## Database connectionv

In [ ]:
#| exporti
db = database(cf.DATABASE_PATH)

## Process events

In [ ]:
#| exporti
process_events = db.create(
    ProcessEvent,
    if_not_exists=True,
    transform=True,
)

In [ ]:
#| export
def log_process_event(
    session_id: str,  # Session receiving the event
    pid: int,         # Operating-system process identifier
    app: str,         # Process or application name
    event_type: str,  # Event type: `start` or `stop`
    timestamp: str,   # ISO-formatted event timestamp
):
    "Record a process start or stop event."
    return process_events.insert(
        session_id=session_id,
        pid=pid,
        app=app,
        event_type=event_type,
        timestamp=timestamp,
    )

In [ ]:
#| export
def get_processes(
    session_id: str,  # Session whose completed process intervals are requested
) -> list[dict]:
    "Return paired process start/stop intervals for a session."
    return db.q(
        """
        SELECT
            s.session_id,
            s.pid,
            s.app,
            s.timestamp AS start_time,
            e.timestamp AS stop_time,
            (julianday(e.timestamp) - julianday(s.timestamp)) * 86400
                AS duration_seconds
        FROM process_event s
        JOIN process_event e
            ON s.pid = e.pid
            AND s.app = e.app
            AND s.session_id = e.session_id
        WHERE s.event_type = 'start'
            AND e.event_type = 'stop'
            AND s.session_id = ?
        """,
        [session_id],
    )

## Monitoring sessions

In [ ]:
#| exporti
sessions = db.create(
    Session,
    pk="session_id",
    if_not_exists=True,
    transform=True,
)

In [ ]:
#| export
def start_session(
    session_id: str,  # Unique identifier for the new session
    start_time: str,  # ISO-formatted session start time
):
    "Create and return a new open monitoring session."
    return sessions.insert(
        session_id=session_id,
        start_time=start_time,
        status="open",
    )

In [ ]:
#| export
def end_session(
    session_id: str,          # Session to close
    end_time: str,            # ISO-formatted session end time
    end_reason: str = "NA",   # Reason the session ended
):
    "Close a monitoring session and record why it ended."
    return sessions.update(
        session_id=session_id,
        end_time=end_time,
        status="closed",
        end_reason=end_reason,
    )

In [ ]:
#| export
def get_last_completed_session() -> dict:
    "Return the most recently completed session, or an empty dictionary."
    rows = db.q(
        """
        SELECT session_id, start_time, end_time
        FROM session
        WHERE status = 'closed'
        ORDER BY julianday(end_time) DESC
        LIMIT 1
        """
    )
    return rows[0] if rows else {}

In [ ]:
#| export
def close_stale_open_sessions(    
    end_time: str,  # Timestamp used to close stale sessions
) -> list[str]:
    "Close sessions left open before startup and return their identifiers."
    open_sessions_dict_list = db.q("""
            SELECT 
                session_id
            FROM session 
            WHERE status = 'open'
            AND julianday(start_time) < julianday(?)
            """, [end_time])
    closed_session_list = []
    if open_sessions_dict_list:
        for open_sessions_dict in open_sessions_dict_list: 
            end_session(session_id = open_sessions_dict['session_id'], end_time = end_time, end_reason = 'interrupted')
            closed_session_list.append(open_sessions_dict['session_id'])
    return closed_session_list

In [ ]:
#| export
def get_session_by_session_id(
    session_id: str,  # Identifier of the requested session
) -> dict:
    "Return a session by ID, using the current time for an open session's end."
    rows = db.q(
        """
        SELECT
            session_id,
            start_time,
            COALESCE(end_time, ?) AS end_time
        FROM session
        WHERE session_id = ?
        ORDER BY julianday(start_time) DESC
        LIMIT 1
        """,
        [
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            session_id,
        ],
    )
    return rows[0] if rows else {}

## Foreground events

In [ ]:
#| exporti
fore_ground_app_events = db.create(
    ForegroundEvent,
    if_not_exists=True,
    transform=True,
)

In [ ]:
#| export
def log_fg_app_events(
    session_id: str,   # Session receiving the interval
    app: str,          # Foreground application name
    pid: int,          # Foreground process identifier
    window_title: str, # Foreground window title
    start_time: str,   # ISO-formatted interval start time
    end_time: str,     # ISO-formatted interval end time
    hwnd: int,         # Windows window handle
    is_idle: bool,     # Whether this was an idle interval
):
    "Record a completed foreground-window or idle interval."
    return fore_ground_app_events.insert(
        session_id=session_id,
        app=app,
        pid=pid,
        window_title=window_title,
        start_time=start_time,
        end_time=end_time,
        hwnd=hwnd,
        is_idle=is_idle,
    )

In [ ]:
#| export
def get_foreground_events(
    session_id: str,  # Session whose foreground intervals are requested
) -> list[dict]:
    "Return a session's foreground intervals in chronological order."
    rows = db.q(
        """
        SELECT *
        FROM foreground_event
        WHERE session_id = ?
        ORDER BY julianday(start_time)
        """,
        [session_id],
    )
    return rows or []

## Data retention

In [ ]:
#| export
def delete_old_data(
    now_ts: str | datetime | None = None,  # Retention reference time; defaults to now
) -> str:
    "Delete data belonging to closed sessions older than the retention period."
    if isinstance(now_ts, str):
        now_dt = datetime.fromisoformat(now_ts)
    else:
        now_dt = now_ts or datetime.now()

    retention_days = getattr(cf, "DATA_RETENTION_DAYS", 365)
    cutoff_ts = (
        now_dt - timedelta(days=retention_days)
    ).isoformat(timespec="seconds")

    db.q(
        """
        DELETE FROM process_event
        WHERE session_id IN (
            SELECT session_id
            FROM session
            WHERE status = 'closed'
              AND julianday(end_time) < julianday(?)
        )
        """,
        [cutoff_ts],
    )

    db.q(
        """
        DELETE FROM foreground_event
        WHERE session_id IN (
            SELECT session_id
            FROM session
            WHERE status = 'closed'
              AND julianday(end_time) < julianday(?)
        )
        """,
        [cutoff_ts],
    )

    db.q(
        """
        DELETE FROM session
        WHERE status = 'closed'
          AND julianday(end_time) < julianday(?)
        """,
        [cutoff_ts],
    )

    return cutoff_ts

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()